In [6]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. تحديد المسارات
file_path = r'C:\Users\Fares ali\Documents\Final_Project\Data\Cleaned_US_Accidents_Sample.csv'
MODEL_DIR = r'..\ModelAPI\models' # فولدر الحفظ الخاص بالـ API

print("⏳ جاري تحميل البيانات الأمريكية النظيفة...")
df_model = pd.read_csv(file_path)

# 2. فصل الميزات (X) عن الهدف (Severity)
X = df_model.drop(columns=['Severity'])
y = df_model['Severity']

# 3. تقسيم البيانات (80% تدريب و 20% اختبار)
print("⏳ جاري تقسيم البيانات (80% تدريب و 20% اختبار)...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. التدريب مع تفعيل الوزن المتوازن (class_weight='balanced')
print("🧠 جاري تدريب موديول الـ Random Forest بالوزن المتوازن...")
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 5. التوقع والتقييم
print("🎯 جاري اختبار الموديول واستخراج النتائج...")
y_pred = rf_model.predict(X_test)

print("\n" + "="*50)
print(f"🎉 دقة الموديول بعد إعادة التوازن (Accuracy): {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("="*50)

print("\n📊 تقرير الأداء التفصيلي لكل مستوى خطورة:")
print(classification_report(y_test, y_pred))

# 6. حفظ الموديل تلقائياً لاستخدامه في الـ API
os.makedirs(MODEL_DIR, exist_ok=True)
model_path = os.path.join(MODEL_DIR, 'road_risk_model.pkl')
joblib.dump(rf_model, model_path)

print(f"✅ تم حفظ الموديل بنجاح في المسار: {model_path}")

⏳ جاري تحميل البيانات الأمريكية النظيفة...
⏳ جاري تقسيم البيانات (80% تدريب و 20% اختبار)...
🧠 جاري تدريب موديول الـ Random Forest بالوزن المتوازن...
🎯 جاري اختبار الموديول واستخراج النتائج...

🎉 دقة الموديول بعد إعادة التوازن (Accuracy): 69.81%

📊 تقرير الأداء التفصيلي لكل مستوى خطورة:
              precision    recall  f1-score   support

           1       0.15      0.07      0.10       859
           2       0.84      0.78      0.81     75597
           3       0.33      0.47      0.38     16980
           4       0.06      0.02      0.03      2555

    accuracy                           0.70     95991
   macro avg       0.34      0.34      0.33     95991
weighted avg       0.72      0.70      0.71     95991

✅ تم حفظ الموديل بنجاح في المسار: ..\ModelAPI\models\road_risk_model.pkl


In [7]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# المسارات الصح بناءً على صورة الفولدر عندك
DATA_PATH = r'C:\Users\Fares ali\Documents\Final_Project\Data\Mock_Incidents_10k_Full.csv'
MODEL_DIR = r'..\ModelAPI\models'

def train_pure_egyptian_model():
    print("⏳ جاري قراءة الداتا المصرية من ملف الـ CSV...")
    df_eg = pd.read_csv(DATA_PATH)
    print(f"📊 حجم الداتا المصرية: {len(df_eg):,} سجل.")
    
    # عرض أسماء الأعمدة للتأكد منها
    print("Columns:", df_eg.columns.tolist())

    # معالجة وتجهيز الأعمدة (حسب شكل الملف)
    # لو الـ Severity نصية بنحولها لأرقام
    if 'Severity' in df_eg.columns and df_eg['Severity'].dtype == object:
        severity_mapping = {'Minor': 1, 'Moderate': 2, 'Major': 3, 'Fatal': 4}
        df_eg['Severity_Encoded'] = df_eg['Severity'].map(severity_mapping).fillna(2).astype(int)
    else:
        df_eg['Severity_Encoded'] = df_eg['Severity']

    # استخراج الساعة ويوم الأسبوع لو التاريخ والوقت موجودين
    if 'Incident_Date' in df_eg.columns and 'Incident_Time' in df_eg.columns:
        df_eg['Incident_DateTime'] = pd.to_datetime(df_eg['Incident_Date'] + ' ' + df_eg['Incident_Time'], errors='coerce')
        df_eg['Hour'] = df_eg['Incident_DateTime'].dt.hour.fillna(12).astype(int)
        df_eg['DayOfWeek'] = df_eg['Incident_DateTime'].dt.dayofweek.fillna(0).astype(int)
    else:
        df_eg['Hour'] = 12
        df_eg['DayOfWeek'] = 0

    # ترميز عمود الـ ADAS لو موجود
    if 'ADAS_Driver_Status' in df_eg.columns:
        le_adas = LabelEncoder()
        df_eg['ADAS_Encoded'] = le_adas.fit_transform(df_eg['ADAS_Driver_Status'].astype(str))
    else:
        df_eg['ADAS_Encoded'] = 0

    # تحديد الميزات المتاحة للتدريب
    available_features = [f for f in ['Hour', 'DayOfWeek', 'Temperature_C', 'Humidity_Pct', 'Wind_Speed_ms', 'CurrentSpeed', 'ADAS_Encoded'] if f in df_eg.columns]
    
    X = df_eg[available_features]
    y = df_eg['Severity_Encoded']

    # تنظيف الـ NaN
    data_clean = pd.concat([X, y], axis=1).dropna()
    X = data_clean[available_features]
    y = data_clean['Severity_Encoded']

    # تقسيم البيانات
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # التدريب
    print("🧠 جاري تدريب الموديول المصري على ملف الـ CSV...")
    egypt_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
    egypt_model.fit(X_train, y_train)

    # التقييم
    y_pred = egypt_model.predict(X_test)
    print("\n" + "="*50)
    print(f"🎉 كفاءة الموديول المصري (Accuracy): {accuracy_score(y_test, y_pred) * 100:.2f}%")
    print("="*50)
    print(classification_report(y_test, y_pred))

    # الحفظ
    os.makedirs(MODEL_DIR, exist_ok=True)
    joblib.dump(egypt_model, os.path.join(MODEL_DIR, 'pure_egypt_model.pkl'))
    print(f"✅ تم حفظ الموديول بنجاح في: {MODEL_DIR}")

if __name__ == "__main__":
    train_pure_egyptian_model()

⏳ جاري قراءة الداتا المصرية من ملف الـ CSV...
📊 حجم الداتا المصرية: 10,000 سجل.
Columns: ['Incident_ID', 'Location_Name', 'Incident_Date', 'Incident_Time', 'Incident_Type', 'Severity', 'Cause', 'Road_Delay_Minutes', 'ADAS_Driver_Status', 'Temperature_C', 'Humidity_Pct', 'Wind_Speed_ms', 'Weather_Condition', 'CurrentSpeed', 'TrafficStatus']
🧠 جاري تدريب الموديول المصري على ملف الـ CSV...

🎉 كفاءة الموديول المصري (Accuracy): 71.55%
              precision    recall  f1-score   support

       Fatal       0.44      0.63      0.52       301
       Major       0.66      0.51      0.57       489
       Minor       0.78      0.73      0.76       422
    Moderate       0.86      0.87      0.86       788

    accuracy                           0.72      2000
   macro avg       0.69      0.69      0.68      2000
weighted avg       0.73      0.72      0.72      2000

✅ تم حفظ الموديول بنجاح في: ..\ModelAPI\models
